# BIOL 339: The command line, BASH, and a real bioinformatics pipeline

**This lab runs across two 3 hour sessions.** By the end of it you will have taken raw DNA sequencing reads and turned them into a biological result, using the same tools a research lab would use.

## Why bother with the command line?

The command line is a text interface to your computer's operating system, and a **shell** is the program that interprets what you type into it. We'll be using the **Bourne Again Shell (BASH)**, which is the default on Linux and the one that nearly all bioinformatics software expects.

There are two practical reasons this is worth your afternoon:

1. **Most bioinformatics software has no graphical interface at all.** If a tool was published in the last twenty years, there's a good chance the only way to run it is from a command line.
2. **The command line scales.** Clicking through 10 files is tedious. Clicking through 10,000 is impossible. A three line BASH script does either without noticing the difference.

For a lot of you this will be the only time in your degree that anyone sits you down in front of a bioinformatics command line. I'm not trying to turn you into programmers. What I want is that when you end up in a lab, or a job, or a summer project, and somebody says "just run this through SPAdes", you know what that means, you aren't frightened of a terminal, and you know how to go and find out the rest.

## What you will actually do

For this lab you are a bioinformatician in a lab that studies the human microbiome. The data are small and synthetic so that everything finishes in lab time, but every tool and every step is real, and the same analysis would scale straight up to real clinical data.

The study: metagenome and RNAseq data were collected from the gut microbiomes of 12 patients, who were then followed for ten years. Six of them stayed healthy. The other six developed colorectal cancer within two years of being sampled. Don't worry though, all of these imaginary patients made a full recovery and then won lotto. Yay!

You have:

- `reads/` : DNA sequencing reads from an exhaustive collection of gut microbes isolated from the patients. In our imaginary world the human gut contains exactly ten microbial species, not thousands.
- `patient_data_1/` and `patient_data_2/` : RNAseq reads from the 12 patients.

Your tasks:

1. Assemble the microbial isolate reads into genomes.
2. Find the genes in each genome and extract them into a single file.
3. Map the patient RNAseq reads against those gene sequences to count transcripts.
4. Use statistics to find genes expressed differently between the healthy and cancer groups, and work out what those genes do.

## How the two sessions are split

| | Content | Roughly |
|---|---|---|
| **Session 1** | Setup, Parts 1 to 5: BASH basics, adapter trimming, genome assembly | 3 hours |
| **Session 2** | Parts 6 to 8: gene finding, read mapping, then the Python notebook | 3 hours |

There is a clearly marked checkpoint at the end of Session 1. If you get there early, there are optional extension exercises. If you do not get there, that is fine, tell a demonstrator and pick up from the checkpoint next time.

## Saving your work

**Ctrl+S**, or the JupyterLab File menu, saves this notebook. Your work lives inside your WSL Ubuntu installation, which persists between sessions on the same machine.

**If you are moving between machines,** or you just want a safety net, email `bash_lab.ipynb` to yourself at the end of each session. To get at the file from Windows, open File Explorer and type `\\wsl$` in the address bar, or run `explorer.exe .` from your Ubuntu terminal.

# Using AI in this lab

You all have free access to AI chatbots, and they are very good at BASH. There is not much point in me pretending otherwise, or asking you not to use them. What matters is which job you give them.

## Two different jobs

<div style="border-left:4px solid #3b5bdb;background:#edf2ff;padding:14px 18px;margin:14px 0">
<b>The exercises are yours.</b> Please don't paste an exercise into a chatbot and copy the answer back. Writing the loop yourself is the whole point of the exercise. It's the bit that transfers to any lab you end up in, and it's the one thing here that you can't pick up just by reading.<br><br>
<b>The AI exercises are the opposite.</b> Three times in this notebook, and once in the python one, I'm going to ask you to get a chatbot to write you something well beyond anything I have taught you, run it on your own data, and then get it to explain how it works. That isn't cheating. It's roughly what I do most weeks.
</div>

This is what using these tools well actually looks like. You want a summary that your data doesn't come with, you know roughly what it should look like, you don't know the incantation, so you ask. The difference between that and just pasting things in and hoping is whether you can read the answer afterwards. So every AI exercise ends by making you read it.

## The routine

The same four steps every time. Steps 2 and 4 are the ones that matter.

<table>
<tr><td width="34"><b>1</b></td><td><b>Ask.</b> Describe your data, say what you want out of it, and say what you have already tried.</td></tr>
<tr><td><b>2</b></td><td><b>Read it before you run it.</b> Say out loud what you think it is going to do. Check the flags it used against <code>toolname --help</code>. If it wants to delete something, be very sure you know what.</td></tr>
<tr><td><b>3</b></td><td><b>Run it, then check the output makes sense.</b> Right number of rows? Numbers a plausible size? A command that runs is not the same thing as a command that is right.</td></tr>
<tr><td><b>4</b></td><td><b>Get it to explain what it did</b>, one piece at a time, then write one sentence in your own words in the notebook. If you can't write the sentence, you haven't finished the exercise.</td></tr>
</table>

## A note on step 2

Command line tools change their option names between versions, sometimes quite a lot. `AdapterRemoval` version 3, for example, renamed `--collapse` to `--merge`, `--trimns` to `--trim-ns`, and `--trimqualities` to `--trim-qualities`. We have pinned version 2 for this lab, so the older names are the right ones here.

A chatbot has read the documentation for all of those versions at once, and it has no way of knowing which one is sitting on your machine. It doesn't know what else you have installed either, or what your files are called. So whatever it gives you, `toolname --help` is the thing that settles it, and that takes about ten seconds.

## Two things not to do

**Don't run a deletion command you haven't read.** `rm` has no undo and no bin. If a chatbot gives you something with `rm` in it, read every character of it before it goes anywhere near your terminal.

**Don't paste patient data into a public AI tool.** Our data are made up, so paste away. Real sequence data from real people is a very different matter, and putting it into a consumer chatbot may well breach your ethics approval, your institution's policy, and in some places the law. Get into the habit now: code and error messages, yes. Data, no.

## When something breaks

And it will. Paste this shape of message rather than "it doesn't work":

```
I am using BASH on Ubuntu.
I ran this command:        <paste the exact command>
I expected:                <what you thought would happen>
I got this error:          <paste the exact error, all of it>
I have already tried:      <what you have tried>
Please explain what the error means and what to check. Do not give me the corrected command yet.
```

That last line is the important one. Ask for the diagnosis first, have a go at it yourself, and ask for the fix only if you still need it.

## How to read this notebook

Cells come in four flavours. Learn to tell them apart:

<table>
<tr><td width="130"><b>Exercise</b></td><td>An empty code cell with a comment telling you what to write. This is the work. Type in it, then run it with <b>Shift+Enter</b>.</td></tr>
<tr><td><b>Check</b></td><td>Run it as-is. It tests whether the previous exercise worked and prints <code>PASS</code> or a message telling you what went wrong. Do not edit these.</td></tr>
<tr><td><b>Helper</b></td><td>Run it as-is. It does something fiddly for you so we can keep moving. Read it anyway, you will usually understand most of it.</td></tr>
<tr><td><b>AI</b></td><td>There are three of these. Each one gets you to use an AI chatbot to write something useful that you couldn't write yet, and then makes you work out how it did it. Have a read of the next section before you hit the first one.</td></tr>
</table>

<div style="border-left:4px solid #c47f00;background:#fff8e6;padding:10px 14px;margin:14px 0">
<b>Note on my spelling.</b> I wrote this in JupyterLab, which has no spell checker, and my spelling is atrocious. Apologies in advance.
</div>

# Part 0: Check your setup

Before this session, you should have worked through the **setup instructions** and created a conda environment called `biol339`.

Two things to confirm:

1. The JupyterLab tab you are reading this in was launched from inside the activated `biol339` environment.
2. The kernel shown in the **top right of this window says `Bash`**. If it says Python, click on it and change it.

Now run the check cell below. It looks for every program this lab needs.

In [20]:
#Check cell: run this as-is. It confirms the tools are installed and reachable.
echo "shell:   $BASH_VERSION"
echo "env:     ${CONDA_DEFAULT_ENV:-NONE (you have not activated biol339)}"
echo
missing=0
for tool in unzip AdapterRemoval spades.py long-orfs extract build-icm glimmer3 kallisto
do
    if command -v $tool > /dev/null 2>&1
    then
        printf "  ok       %s\n" "$tool"
    else
        printf "  MISSING  %s\n" "$tool"
        missing=1
    fi
done
echo
if [ $missing -eq 0 ]
then
    echo "PASS: all tools found. You are ready to start."
else
    echo "FAIL: something is missing. Check that you activated the biol339 environment"
    echo "      BEFORE launching jupyter lab, then re-read the setup instructions."
fi

shell:   5.3.9(1)-release
env:     biol339

  ok       unzip
  ok       AdapterRemoval
  ok       spades.py
  ok       long-orfs
  ok       extract
  ok       build-icm
  ok       glimmer3
  ok       kallisto

PASS: all tools found. You are ready to start.


<div style="border-left:4px solid #c92a2a;background:#fff5f5;padding:12px 16px;margin:14px 0">
<b>If that said FAIL, stop and fix it now.</b> Nothing later in the lab will work. Nine times out of ten it is because JupyterLab was launched from the wrong place. Close JupyterLab, go back to your Ubuntu terminal, and run:
<pre>conda activate biol339
cd ~/339_lab-main
jupyter lab</pre>
The environment name should appear in brackets at the start of your terminal prompt, like <code>(biol339) you@machine:~$</code>, before you launch JupyterLab.
</div>

---
# Session 1
---

# Part 1: Finding your way around

A graphical file manager and the command line do the same job. One shows you the files, the other tells you about them. The directory you are currently sitting in is your **working directory**, and almost every command you type is interpreted relative to it. Knowing where you are is the single most useful habit in this lab.

`pwd` **p**rints the **w**orking **d**irectory.

**Exercise 1:** In the cell below, use `pwd` to print your working directory.

In [21]:
#add code for exercise 1 below this comment:

pwd

/home/Alice/339_lab-main


`cd` **c**hanges **d**irectory. `ls` **l**i**s**ts what is in a directory.

The tilde `~` is a shortcut for your home directory, so `~/339_lab-main` means "the folder `339_lab-main` inside my home directory", no matter where you currently are. If you ever get lost, `cd ~` takes you home.

**Exercise 2:** In the cell below, use `cd` to change to `~/339_lab-main`, then use `ls` to list what is in it. Put both commands in the same cell, on separate lines. They run one after the other.

In [22]:
#add code for exercise 2 below this comment:

cd ~/339_lab-main

ls

bash_lab.ipynb  patient_data_1  python_lab.ipynb  setup.html
patient_data    patient_data_2  reads


In [23]:
#Check cell for exercise 2: run this as-is.
if [ "$PWD" = "$HOME/339_lab-main" ]
then
    echo "PASS: you are in $PWD and it contains:"
    ls
else
    echo "FAIL: you are in $PWD"
    echo "      You should be in $HOME/339_lab-main"
fi

PASS: you are in /home/Alice/339_lab-main and it contains:
bash_lab.ipynb  patient_data_1  python_lab.ipynb  setup.html
patient_data    patient_data_2  reads


**Expected contents**

```
bash_lab.ipynb   patient_data_1.zip   patient_data_2.zip   python_lab.ipynb   reads.zip   setup.html
```

<div style="border-left:4px solid #3b5bdb;background:#edf2ff;padding:12px 16px;margin:14px 0">
<b>Working directory and the notebook.</b> Every cell in this notebook shares one BASH session, so a <code>cd</code> in one cell still applies in the next. That is convenient, but it also means that if you skip a cell or run cells out of order, you can end up somewhere you did not expect. When a command fails for no obvious reason, <b>run <code>pwd</code> first.</b> It is the answer surprisingly often.
</div>

## Wildcards

`*` is a **wildcard** that matches any run of characters. The shell expands it into a list of matching file names *before* the command runs.

So `ls *.zip` does not pass `*.zip` to `ls`. The shell first works out which files match, then hands `ls` the actual list. This is worth understanding because it explains a lot of otherwise baffling behaviour later.

Try it:

In [24]:
ls *.zip

ls: cannot access '*.zip': No such file or directory


: 2

## Unpacking the data

`unzip` unpacks a `.zip` file. The name of the file goes after the command, separated by a space:

```bash
unzip my_file.zip
```

<div style="border-left:4px solid #3b5bdb;background:#edf2ff;padding:12px 16px;margin:14px 0">
<b>Why no spaces in file names?</b> BASH uses spaces to separate arguments. A file called <code>my data.fq</code> looks to BASH like two arguments, <code>my</code> and <code>data.fq</code>. Everything in this lab uses underscores instead. Do the same with your own data forever, your future self will thank you.
</div>

**Exercise 3:** Write commands to unzip all three zip files. You will need **three separate lines**, one per file.

Why not `unzip *.zip`? Because `unzip` treats its second argument as a *filter for files inside the archive*, not as another archive. After the shell expands the wildcard, `unzip` sees `unzip patient_data_1.zip patient_data_2.zip reads.zip` and tries to extract members named `patient_data_2.zip` and `reads.zip` from inside `patient_data_1.zip`. This is a good early example of a command that fails in a confusing way for a completely logical reason.

<div style="border-left:4px solid #c47f00;background:#fff8e6;padding:10px 14px;margin:14px 0">
<b>Do not unzip the same file twice.</b> <code>unzip</code> will stop and ask you whether to overwrite, and because this is a notebook you cannot answer it. The cell will hang forever. If that happens, restart the kernel from the <b>Kernel</b> menu at the top of the page, then carry on.
</div>

In [ ]:
#add code for exercise 3 below this comment:

unzip patient_data_1.zip

unzip patient_data_2.zip

unzip reads.zip


Archive:  patient_data_1.zip


In [25]:
#Check cell for exercise 3: run this as-is.
ok=1
for d in reads patient_data_1 patient_data_2
do
    n=$(ls $d/*.fq 2>/dev/null | wc -l)
    printf "  %-16s %s .fq files\n" "$d" "$n"
    [ "$n" -gt 0 ] || ok=0
done
[ $ok -eq 1 ] && echo "PASS: all three archives unpacked." \
              || echo "FAIL: at least one folder is missing or empty."

  reads            20 .fq files
  patient_data_1   12 .fq files
  patient_data_2   12 .fq files
PASS: all three archives unpacked.


**Expected output**

```
  reads            20 .fq files
  patient_data_1   12 .fq files
  patient_data_2   12 .fq files
PASS: all three archives unpacked.
```

<div style="border-left:4px solid #3b5bdb;background:#edf2ff;padding:12px 16px;margin:14px 0">
<b>What is <code>__MACOSX</code>?</b> If you run <code>ls</code> you will see a folder called <code>__MACOSX</code> appear. It is junk that macOS adds when it makes a zip file, and it is harmless. You can delete it in the next exercise.
</div>

## Deleting things

`rm` **r**e**m**oves files. Names go after the command, separated by spaces:

```bash
rm junk_1.txt junk_2.txt junk_3.txt
```

Wildcards work too. `rm *.txt` removes every `.txt` file in the working directory, and `rm *junk*` removes every file with `junk` anywhere in its name.

<div style="border-left:4px solid #c92a2a;background:#fff5f5;padding:12px 16px;margin:14px 0">
<b>There is no undo and no recycle bin.</b> Deleted is deleted. This matters most with wildcards, because you are trusting a pattern rather than reading a list.<br><br>
<b>The habit that saves you:</b> run <code>ls</code> with your pattern first, look at what comes back, and only then swap <code>ls</code> for <code>rm</code>. Do this every single time. Professionals do this every single time.
</div>

**Exercise 4:** Free up some space.

1. First run `ls *.zip` to see what you are about to delete.
2. Then delete the three zip files with a single `rm` command using a wildcard.
3. Then delete the `__MACOSX` folder. Folders need `rm -r`, where `-r` means recursive: `rm -r __MACOSX`

In [3]:
#add code for exercise 4 below this comment:

ls *.zip

rm *.zip

rm -r __MACOSX

patient_data_1.zip  patient_data_2.zip  reads.zip


In [26]:
#Check cell for exercise 4: run this as-is.
z=$(ls *.zip 2>/dev/null | wc -l)
m=$(ls -d __MACOSX 2>/dev/null | wc -l)
echo "zip files remaining: $z    __MACOSX folders remaining: $m"
echo "directories now here:"
ls -d */
[ "$z" -eq 0 ] && [ "$m" -eq 0 ] && echo "PASS" || echo "FAIL: see the two counts above, both should be 0."

zip files remaining: 0    __MACOSX folders remaining: 0
directories now here:
patient_data/  patient_data_1/  patient_data_2/  reads/
PASS


## Making and moving

`mkdir` **m**a**k**es **dir**ectories. It takes as many names as you like at once:

```bash
mkdir dir1 dir2 dir3
```

`mv` **m**o**v**es files, and also renames them, which is the same operation as far as the filesystem is concerned.

- `mv *.txt ..` moves every `.txt` file one level up. `..` always means "the directory one level up".
- `mv *.txt ../new_location` moves them into a folder called `new_location` one level up.
- `mv old_name.txt new_name.txt` renames a file.

**Exercise 5:** Our patient data arrived split across two folders for no good reason. Put it back together.

1. Use `mkdir` to make a directory called `patient_data`.
2. Use `mv` with a wildcard to move everything out of `patient_data_1` and `patient_data_2` into it.
3. The two now-empty folders can go: `rm -r patient_data_1 patient_data_2`

**Hint:** `patient_data_*/*` matches every file inside any folder whose name starts with `patient_data_`.

In [14]:
#add code for exercise 5 below this comment:

mkdir patient_data

mv patient_data_*/*.fq ../patient_data 

rm -r patient_data_1 patient_data_2

mv: cannot stat 'patient_data_*/*.fq': No such file or directory
rm: cannot remove 'patient_data_1': No such file or directory
rm: cannot remove 'patient_data_2': No such file or directory


: 1

In [ ]:
#Check cell for exercise 5: run this as-is.
n=$(ls patient_data/*.fq 2>/dev/null | wc -l)
echo "files in patient_data: $n  (expected 24)"
ls -d */
[ "$n" -eq 24 ] && echo "PASS" || echo "FAIL: expected 24 .fq files in patient_data."

## Counting things

Here is a command you have already seen in the check cells:

```bash
ls patient_data/* | wc -l
```

The `|` symbol is a **pipe**. It takes the output of the command on its left and feeds it as input to the command on its right, instead of printing it to the screen. `wc -l` counts lines. So this says: list the files, then count how many lines that list had.

We will use pipes properly in Part 2. For now, just note that `ls | wc -l` is how you count files.

**Exercise 6:** Modify that command so it counts only files ending in `.fq`.

In [13]:
#add code for exercise 6 below this comment:

cd ~/339_lab-main/patient_data

ls *.fq | wc -l

24


**Expected output**

```
24
```

## Patterns with more than one wildcard

Run the cell below to move into the `reads` folder and look at what is in it.

Notice that files come in pairs: `..._reads1.fq` and `..._reads2.fq`. These are the **forward** and **reverse** reads from paired-end sequencing of ten different genomes. The sequencer reads each DNA fragment from both ends, so read 1 and read 2 in a pair come from the same fragment and belong together. Nearly every tool you will use wants both files, and wants them matched up correctly.

In [19]:
cd ~/339_lab-main/reads
ls

genome_10_reads1.fq  genome_2_reads2.fq  genome_5_reads1.fq  genome_7_reads2.fq
genome_10_reads2.fq  genome_3_reads1.fq  genome_5_reads2.fq  genome_8_reads1.fq
genome_1_reads1.fq   genome_3_reads2.fq  genome_6_reads1.fq  genome_8_reads2.fq
genome_1_reads2.fq   genome_4_reads1.fq  genome_6_reads2.fq  genome_9_reads1.fq
genome_2_reads1.fq   genome_4_reads2.fq  genome_7_reads1.fq  genome_9_reads2.fq


**Exercise 7:** Write a command that lists **only the forward reads**.

**Hint:** you can use `*` more than once in a pattern. `ls *something*.txt` lists every `.txt` file with `something` somewhere in its name.

In [20]:
#add code for exercise 7 below this comment:

ls *reads1.fq

#i put *1.fq which also works :c

genome_10_reads1.fq  genome_3_reads1.fq  genome_6_reads1.fq  genome_9_reads1.fq
genome_1_reads1.fq   genome_4_reads1.fq  genome_7_reads1.fq
genome_2_reads1.fq   genome_5_reads1.fq  genome_8_reads1.fq


In [21]:
#Check cell for exercise 7: run this as-is.
n=$(ls *reads1*.fq 2>/dev/null | wc -l)
echo "forward read files: $n  (expected 10)"
[ "$n" -eq 10 ] && echo "PASS" || echo "FAIL"

forward read files: 10  (expected 10)
PASS


## Three more patterns

`*` is not the only wildcard. `?` matches **exactly one** character, and `[0-9]` matches **exactly one** character from the set given.

**Predict before you run.** Write down how many files you think each of these will list, then run the cell and see.

```bash
ls genome_?_reads1.fq
ls genome_[0-9]_reads1.fq
ls genome_1*_reads1.fq
```

The third one catches almost everybody, and the reason it does is worth remembering.

In [25]:
#try the three patterns here, after writing down your predictions:

#first will list 9 {correct}
#second will list 9 {correct}
#third is going to list 2. {also correct}

ls genome_?_reads1.fq

ls genome_[0-9]_reads1.fq

ls genome_1*_reads1.fq


genome_10_reads1.fq  genome_1_reads1.fq


---
# Part 2: Looking inside files

You now have 44 sequencing files and no idea what is in any of them. Being able to peer into a large file without opening it is one of the things the command line is really good at, and it is how you catch a corrupted download or a misnamed sample before you waste an hour assembling nonsense.

Four commands do most of the work:

| Command | What it does |
|---|---|
| `head -n 8 file` | print the first 8 lines |
| `tail -n 8 file` | print the last 8 lines |
| `wc -l file` | count the lines |
| `grep pattern file` | print only lines containing `pattern` |

Run the cell below to look at the first two reads in one of our files.

In [26]:
head -n 8 genome_1_reads1.fq

@1-21800/1
TTACGGAGAAGAGAATACGGGATATCGTTTGATAAAATTCCCTTGGTTACCTCAATCGAAAGATTTGTTCCAAATGAACTCTATTCCAGAGATCAGATTGGCAGAGATTTACTATTCGTTAGCTGAATGTAAATATAGATCAGGGGATAA
+
FAFFFJJJJJJJJAJJJJJJJJJJ<JJJJJJJJJJJJJJJJJFJJJJJJFJ,<JJJFJJJJJJJJJJJJJJJJJFFJJJJJJJJJJJJJJJJJJJJJJ<JJFFFJJJJJJJ,JJJJJJJJJFJFAJJFJAJ,JFJJJJJFJFJJJJJAJ7
@1-21798/1
GGAGAAGCGGGAGAGAGGGGAAATGATTGGATCTTAATTTTGGAGGCGGATGATAGGTAGACGATGTATAAATTTTAAATAATAAAAGTCATTATGAAGAAGATCAATGTTTTACTGTTAGCCTTTTTGCTGTTAAGTTTTTGGGCGAAT
+
FAFFFJJJJJJJJJJJJJJJJJJJ7JJJJJJJJJJAJJJFJJJJJJJFJJJJJFJJJFJJJJJJJJJJJJJJJJJJFJJFJJJJJJJJJJJJJJJJJJFJJJJJJJJJJJJJJJJ<JJJ,<AJJJJFJJJFJJJJJJJJ,JJJJJFJAJ,


## The FASTQ format

That output is **FASTQ**, the standard format for sequencing reads. Every read takes exactly **four lines**:

1. `@` followed by the read name
2. the DNA sequence
3. `+` (a separator, sometimes repeating the name)
4. a quality score for each base, one character per base

That "exactly four lines" is the useful part. `wc -l` tells you the number of lines, so the number of reads is the number of lines divided by four.

**Exercise 8:** Use `wc -l` to count the lines in `genome_1_reads1.fq`, and work out how many reads that is. You can do the division in your head, or use BASH arithmetic: `echo $(( 1000 / 4 ))` prints 250.

In [ ]:
#add code for exercise 8 below this comment:

**Expected output:** 43600 lines, which is 10900 reads.

## Pipes and redirection

Two symbols do an enormous amount of work in BASH.

**The pipe `|`** sends the output of one command into the next. Commands become building blocks you can chain:

```bash
grep "^@" reads.fq | wc -l        # count lines starting with @
```

**Redirection `>`** sends output into a file instead of onto the screen:

```bash
ls *.fq > file_list.txt           # write the list into a file
ls *.fq >> file_list.txt          # append to the file instead of overwriting
```

<div style="border-left:4px solid #c47f00;background:#fff8e6;padding:10px 14px;margin:14px 0">
<code>&gt;</code> <b>overwrites without warning.</b> <code>command &gt; results.txt</code> silently destroys whatever <code>results.txt</code> used to contain. Use <code>&gt;&gt;</code> when you mean "add to".
</div>

There is a third one you will meet in Part 5. Programs write their normal output to a channel called **stdout** and their error and progress messages to a separate channel called **stderr**. `>` captures only stdout. `2>&1` means "send stderr to the same place as stdout", so `command > log.txt 2>&1` captures absolutely everything. Genome assemblers are extremely chatty, and this is how you stop them flooding your screen.

**Exercise 9:** Use `grep` and a pipe to count how many reads in `genome_1_reads1.fq` contain the sequence `GGGGG` (five Gs in a row).

**Hint:** `grep -c pattern file` counts matching lines directly, so you do not strictly need the pipe. Try it both ways: `grep pattern file | wc -l` and `grep -c pattern file`. They should agree.

In [ ]:
#add code for exercise 9 below this comment:

<div style="border-left:4px solid #6c757d;background:#f1f3f5;padding:12px 16px;margin:14px 0">
<b>Extension (optional, skip if you are short on time).</b> That count is not quite "how many reads contain GGGGG", it is "how many <i>lines</i> contain GGGGG". Since a quality score line can also contain the letter G, some of those matches are not sequence at all. Can you think of a way to search only the sequence lines? You do not have to implement it, but if you want a hint, ask your AI chatbot to explain what <code>awk 'NR%4==2'</code> does. This is exactly the kind of subtle mistake that makes bioinformatics results wrong in ways nobody notices.
</div>

**Exercise 10:** Save a list of all the forward read file names into a file called `forward_reads.txt`, then use `cat` to print the file's contents to check it worked. (`cat` prints a whole file to the screen.)

In [ ]:
#add code for exercise 10 below this comment:

In [ ]:
#Check cell for exercise 10: run this as-is.
if [ -f forward_reads.txt ]
then
    echo "PASS: forward_reads.txt has $(wc -l < forward_reads.txt) lines"
else
    echo "FAIL: forward_reads.txt was not created"
fi
rm -f forward_reads.txt

<div style="border-left:4px solid #3b5bdb;background:#edf2ff;padding:16px 20px;margin:16px 0">
<h3 style="margin:0 0 10px">AI exercise 1: a read count table</h3>

<b>The job.</b> You have just counted the reads in <i>one</i> file, by hand, with <code>wc -l</code> and a division. You have <b>twenty</b> files. Nobody does that twenty times.<br><br>

What you want is a table of every file name and how many reads are in it. It's the first thing anyone looks at when sequencing data turns up, and you can't write it yet, because it needs a bit of BASH we haven't covered.<br><br>

<b>1. Ask.</b> Something along these lines:<br>
<i>"I am in a directory with 20 FASTQ files ending in .fq. Every read takes exactly 4 lines. Give me one BASH command that prints a two column table of the file name and the number of reads in it. I know about loops and wc, but not much else."</i><br><br>

<b>2. Read it before you run it.</b> You will most likely get something built around <code>$( )</code> and <code>$(( ))</code>, neither of which you have seen. Don't run it yet. Have a look and see if you can work out which bit does the counting and which bit does the dividing.<br><br>

<b>3. Run it</b> in the cell below and look at what comes back. Twenty rows? Numbers in the ten thousands? Does <code>genome_1_reads1.fq</code> come out as 10900, the same as you got by hand in exercise 8? If it doesn't, something is wrong, and this is exactly why step 3 is there.<br><br>

<b>4. Get it to explain.</b> Ask these one at a time:<br>
<i>"What is the difference between <code>$( )</code> and <code>$(( ))</code> here?"</i><br>
<i>"Why <code>wc -l &lt; file</code> rather than <code>wc -l file</code>? What changes in the output?"</i><br><br>

That second one has a nice answer, and it explains a small mystery you would otherwise trip over later on.<br><br>

<b>Write it down.</b> In the cell below your command, add a comment of one sentence, in your own words, saying what the command does.
</div>

In [ ]:
#AI exercise 1: paste the command here, then add your one sentence explanation as a comment.

---
# Part 3: Variables, arrays, and loops

This is the part that turns "typing commands" into "programming", and it is the core skill of the whole lab. Everything after this is an application of it.

## Variables

BASH assigns variables with `=` and **no spaces around it**:

In [ ]:
course=biol339
echo $course

`echo` prints something to the screen. The dollar sign `$` means **"the value of"**, so `$course` means the value stored in `course`.

<div style="border-left:4px solid #c47f00;background:#fff8e6;padding:10px 14px;margin:14px 0">
<b>The spaces rule.</b> <code>course=biol339</code> works. <code>course = biol339</code> does not, and gives you a baffling error, because BASH reads it as "run the command <code>course</code> with the arguments <code>=</code> and <code>biol339</code>". This trips up everybody at least once.
</div>

Variables can be dropped into the middle of any command, including inside a sentence:

In [ ]:
echo I love $course

In JupyterLab, a variable you set in one cell is still there in later cells. But **if you restart the kernel, every variable is wiped** and you have to set them again. There is a "restore state" cell at the start of Session 2 for exactly this reason.

## Arrays

An array is an ordered list of values. Make one with parentheses, with a **space** between elements:

```bash
plagues=(blood frogs lice flies sickness boils hail locusts darkness death)
```

To get things back out you need `${ }` around the name, and square brackets for the position:

```bash
echo ${plagues[0]}     # blood      <- positions count from ZERO
echo ${plagues[3]}     # flies
echo ${plagues[*]}     # every element
echo ${#plagues[*]}    # 10, the number of elements
```

Position counting starts at **0**, so the first element is `[0]` and the tenth is `[9]`. Nearly every programming language does this and nearly everyone gets caught by it occasionally.

You can also change one element in place:

```bash
plagues[4]=disease
```

<div style="border-left:4px solid #6c757d;background:#f1f3f5;padding:10px 14px;margin:14px 0">
That array section is adapted from <i>The Unix Workbench</i> by Sean Kross, which is free online and is a good next step if you want to take this further.
</div>

**The useful bit for us:** a wildcard pattern in parentheses builds an array of every matching file name. Run the cell below to make an array of all the forward read files.

In [ ]:
fw_reads=(*reads1*)
echo ${fw_reads[*]}

**Exercise 11:** In the cell below, print the **number of elements** in `fw_reads`.

In [ ]:
#add code for exercise 11 below this comment:

**Expected output:** `10`

**Exercise 12:** Print the first, second, third, and tenth elements of `fw_reads`, one per line. Your answer will be **four lines**.

Note the order they come out in. `genome_10` sorts before `genome_2` because the shell sorts these as text, not as numbers, and `1` comes before `2`. This is not a mistake, and it will matter in a moment.

In [ ]:
#add code for exercise 12 below this comment:

**Expected output**

```
genome_10_reads1.fq
genome_1_reads1.fq
genome_2_reads1.fq
genome_9_reads1.fq
```

**Exercise 13:** Make a second array called `rv_reads` containing all the **reverse** read files. They contain the pattern `reads2`.

In [ ]:
#add code for exercise 13 below this comment:

In [ ]:
#Check cell for exercise 13: run this as-is.
echo "length:      ${#rv_reads[*]}   (expected 10)"
echo "last element: ${rv_reads[9]}   (expected genome_9_reads2.fq)"
[ "${#rv_reads[*]}" -eq 10 ] && [ "${rv_reads[9]}" = "genome_9_reads2.fq" ] \
    && echo "PASS" || echo "FAIL"

## Why two arrays matters

Both arrays were built by the shell from the same sort of pattern, so they came out in the **same order**. Position 2 in `fw_reads` and position 2 in `rv_reads` are the two halves of the same sample:

In [ ]:
echo ${fw_reads[2]} and ${rv_reads[2]} are at position 2

That is the trick the entire rest of this lab depends on. Paired-end tools need matched forward and reverse files. Two arrays plus a shared position gives you that, for any number of samples, without you ever typing a file name.

## For loops

A `for` loop repeats a block of commands once for each item in a list.

In [ ]:
for i in ${fw_reads[@]}
do
    echo $i
done

Reading that line by line:

- `for i in ${fw_reads[@]}` sets up the loop. `${fw_reads[@]}` is the list being stepped through, and `i` is the name given to the current item. On the first pass `i` holds `genome_10_reads1.fq`, on the second pass it holds `genome_1_reads1.fq`, and so on. `i` is just a name. You could call it `j`, `read`, or `cheeseburger`.
- Everything between `do` and `done` runs once per item.
- The indentation is only for readability. BASH does not care.

You can write the same loop on one line using `;` as a separator, which is handy for quick one-offs:

In [ ]:
for i in ${fw_reads[@]}; do echo $i; done

## Looping over positions instead of values

Adding an exclamation mark, `${!fw_reads[@]}`, gives you the **positions** (0, 1, 2, ...) instead of the values.

In [ ]:
for i in ${!fw_reads[@]}
do
    echo $i
done

And once you have the position, you can pull the item at that position out of the array:

In [ ]:
for i in ${!fw_reads[@]}
do
    echo ${fw_reads[$i]}
done

That looks like a pointlessly complicated way to get the same answer as before. It is not, and here is why: **a position can be used to index into more than one array at once**. That is how you keep forward and reverse reads together.

**Exercise 14:** Write a for loop that steps through the **positions** of `fw_reads` and, for each one, prints a sentence naming the forward file, the reverse file, and the position, in the format shown below.

In [ ]:
#add code for exercise 14 below this comment:

**Expected output**

```
genome_10_reads1.fq and genome_10_reads2.fq are at position 0
genome_1_reads1.fq and genome_1_reads2.fq are at position 1
genome_2_reads1.fq and genome_2_reads2.fq are at position 2
genome_3_reads1.fq and genome_3_reads2.fq are at position 3
genome_4_reads1.fq and genome_4_reads2.fq are at position 4
genome_5_reads1.fq and genome_5_reads2.fq are at position 5
genome_6_reads1.fq and genome_6_reads2.fq are at position 6
genome_7_reads1.fq and genome_7_reads2.fq are at position 7
genome_8_reads1.fq and genome_8_reads2.fq are at position 8
genome_9_reads1.fq and genome_9_reads2.fq are at position 9
```

Look carefully at that output. Every forward file is matched with the correct reverse file, and you never typed a single file name. If there were 400 samples instead of 10, the loop would be **exactly the same**. That is the whole point.

<div style="border-left:4px solid #c47f00;background:#fff8e6;padding:14px 18px;margin:16px 0">
<b>A two minute detour: spaces around the equals sign</b><br><br>
Run the cell below. It is meant to fail, and it is worth seeing it fail now rather than at 4pm on a deadline.<br><br>
Read the error carefully. It won't look anything like the mistake you made, because BASH has no idea you were trying to set a variable. It saw a word, then a space, and assumed you wanted to run a command by that name. This is why I made a fuss about the spaces a moment ago.<br><br>
Fix it and run it again. If you want the practice, put the error through the debugging template from the top of this notebook first and see what you get back.
</div>

In [ ]:
#Run this cell as-is. It is MEANT to fail. Then fix it.
my_file = genome_1_reads1.fq
echo "the file is $my_file"

---
# Part 4: Trimming adapter sequences

You are now going to run your first real bioinformatics program.

**Command line applications** are programs you launch by typing their name. They take **arguments** that tell them which files to work on and what to do with them. Arguments usually come in the form `--name value`, and the ones starting with `--` or `-` are often called **flags** or **options**.

## Why trim?

During library preparation, short synthetic DNA sequences called **adapters** are ligated onto the ends of every fragment so the sequencer can grab hold of them. If a fragment is shorter than the read length, the sequencer runs off the end of the real DNA and starts reading adapter. Leave that in and your assembler will try to assemble adapter sequence, which produces garbage.

`AdapterRemoval` finds and strips adapters, trims low quality bases from the ends, and writes out new, cleaner read files.

To trim one pair of files, the command is:

```bash
AdapterRemoval --file1 fwd_reads.fq --file2 rv_reads.fq --basename new_file_name \
--trimns --trimqualities --collapse
```

| Argument | Meaning |
|---|---|
| `--file1` | the forward reads |
| `--file2` | the reverse reads |
| `--basename` | the prefix for all the output files it will create |
| `--trimns` | trim ambiguous bases (`N`) from the ends |
| `--trimqualities` | trim low quality bases from the ends |
| `--collapse` | where a forward and reverse read overlap, merge them into one longer, more accurate read |

The `\` at the end of the first line is a **line continuation**. It lets you split one long command across two lines for readability. There must be nothing after it, not even a space.

In **exercises 15 to 18** we build the loop one piece at a time. Do not skip ahead to the finished version. Each step is one small idea, and building it this way is how you will write your own scripts later.

**Exercise 15:** Write a for loop that steps through the positions of `fw_reads` and, at each position, prints the forward file name and the reverse file name **on separate lines**.

(This is almost exercise 14 again, deliberately. Start from something that works.)

In [ ]:
#add code for exercise 15 below this comment:

**Expected output**

```
genome_10_reads1.fq
genome_10_reads2.fq
genome_1_reads1.fq
genome_1_reads2.fq
...and so on for all ten pairs
```

**Exercise 16:** Update your loop so that instead of echoing the array elements directly, it first stores them in two variables called `reads_1` and `reads_2`, then echoes those two variables on one line.

This feels like a pointless extra step. It is not: in a moment those variable names are what you will hand to the program, and giving things meaningful names is what makes a script readable six months later.

In [ ]:
#add code for exercise 16 below this comment:

In [ ]:
#Check cell for exercise 16: run this as-is.
echo "after the loop, reads_1 = $reads_1"
echo "after the loop, reads_2 = $reads_2"
[ "$reads_1" = "genome_9_reads1.fq" ] && [ "$reads_2" = "genome_9_reads2.fq" ] \
    && echo "PASS" || echo "FAIL: expected the last pair, genome_9_reads1.fq and genome_9_reads2.fq"

<div style="border-left:4px solid #3b5bdb;background:#edf2ff;padding:12px 16px;margin:14px 0">
Why does <code>$reads_1</code> still have a value after the loop finished? Because a loop variable is not tidied away at <code>done</code>. It simply keeps whatever it held on the final pass. That is occasionally useful and occasionally the cause of a very confusing bug.
</div>

## Building an output name from an input name

`AdapterRemoval` needs a `--basename`, a prefix for the files it writes. We want `genome_1_reads1.fq` to give us `genome_1`.

BASH can chop the end off a variable's value. This is called **parameter expansion**:

```bash
genome_name=${reads_1%_reads1.fq}
```

The `%` means "delete the shortest match of this pattern **from the end**". So if `reads_1` holds `genome_7_reads1.fq`, then `${reads_1%_reads1.fq}` gives `genome_7`.

You may also see this written with question marks, where each `?` matches exactly one character:

```bash
genome_name=${reads_1%??????????}
```

`_reads1.fq` is ten characters, so ten question marks removes it. Both versions work. **The first is far better**, because it says what it means, and because it does not silently break the day someone hands you a file called `sample_reads1.fastq`.

**Exercise 17:** Update your loop so that it also sets a variable called `genome_name`, then prints all three variables on one line.

In [ ]:
#add code for exercise 17 below this comment:

In [ ]:
#Check cell for exercise 17: run this as-is.
echo "reads_1 = $reads_1"
echo "reads_2 = $reads_2"
echo "genome_name = $genome_name"
[ "$genome_name" = "genome_9" ] && echo "PASS" || echo "FAIL: genome_name should be genome_9"

**Expected loop output**

```
genome_10_reads1.fq genome_10_reads2.fq genome_10
genome_1_reads1.fq genome_1_reads2.fq genome_1
genome_2_reads1.fq genome_2_reads2.fq genome_2
...and so on
```

<div style="border-left:4px solid #3b5bdb;background:#edf2ff;padding:16px 20px;margin:16px 0">
<h3 style="margin:0 0 10px">AI exercise 2: better trimming</h3>

<b>The job.</b> The command you are about to run does the basics. A proper trimming step usually does two more things, and <code>AdapterRemoval</code> can do both of them, but I haven't told you what the options are called.<br><br>

The two things are <b>sliding window quality trimming</b>, which walks along the read and cuts where the average quality drops rather than only tidying up the ends, and <b>a minimum length filter</b>, which throws away reads that are left too short to be any use after trimming.<br><br>

<b>1. Ask.</b> Give it the command you already have and ask for the additions:<br>
<i>"I am running AdapterRemoval with <code>--file1 --file2 --basename --trimns --trimqualities --collapse</code>. Which options add sliding window quality trimming, and which one discards reads shorter than 50 bases after trimming? Give me the option names and the values they take."</i><br><br>

<b>2. Check the options against your own copy</b> before you use any of them. Run the cell below, which searches the help text of the <code>AdapterRemoval</code> that is actually installed on your machine. If an option it suggested is in there, with the same name, you are fine. If it isn't, ask the chatbot what the equivalent is in version 2, or search the help text yourself with a different word.<br><br>

As I said in the section at the top, tools rename their options between versions, and the chatbot has no way of knowing which version you have. This is not something to be annoyed about, it is just a thing to check, and checking takes ten seconds. Get in the habit of doing it every time and it stops being a problem.<br><br>

<b>3 and 4.</b> Once you have option names your copy recognises, add them to your loop in exercise 18 and run it. Then ask: <i>"What does a sliding window quality trim actually do to a read, and why is it better than only trimming the ends?"</i> Write the answer as a one sentence comment in your exercise 18 cell.
</div>

In [ ]:
#AI exercise 2: check the options against the AdapterRemoval you actually have.
#Add any other words you want to search for to the list inside the quotes.
AdapterRemoval --version
echo
echo "----- options matching those words -----"
AdapterRemoval --help 2>&1 | grep -i -E "minlength|trimwindows|minquality|trim-window|min-length"

**Exercise 18:** Now replace `echo` with the real thing.

Update your loop so that instead of printing the three variables, it passes them to `AdapterRemoval`:

- `--file1` gets `$reads_1`
- `--file2` gets `$reads_2`
- `--basename` gets `$genome_name`
- and add `--trimns --trimqualities --collapse` as in the example

It will print a lot of output. That is normal.

In [ ]:
#add code for exercise 18 below this comment:

In [ ]:
#Check cell for exercise 18: run this as-is.
n=$(ls *pair*.truncated 2>/dev/null | wc -l)
echo "trimmed paired read files: $n  (expected 20)"
[ "$n" -eq 20 ] && echo "PASS" || echo "FAIL: expected 20 files matching *pair*.truncated"

## Reading what a tool tells you

`AdapterRemoval` wrote a `.settings` file for each genome, recording exactly what it did. **Reading these files is a real skill.** A step that silently did nothing, or silently threw away 90% of your data, looks identical to a successful step unless you check.

**Exercise 19:** Print `genome_1.settings` and find the section headed `[Trimming statistics]`.

`cat` prints a whole file, but this one is long, so pipe it into `grep` and pull out just the section you want:

```bash
grep -A 14 "Trimming statistics" genome_1.settings
```

`-A 14` means "and the 14 lines after each match". Look at `Number of reads with adapters` and at `Number of full-length collapsed pairs`.

In [ ]:
#add code for exercise 19 below this comment:

<div style="border-left:4px solid #c47f00;background:#fff8e6;padding:12px 16px;margin:14px 0">
<b>What you should notice, and why it is fine.</b> <code>Number of reads with adapters</code> is <b>0</b>. Not "a few". None. That is because these reads were simulated and I did not simulate any adapter contamination into them, so on this dataset the adapter trimming step does nothing whatsoever. (<code>--collapse</code> did do something: a hundred or so overlapping pairs were merged.)<br><br>
So why make you run it? Three reasons. With real Illumina data it is never a no-op, and skipping it produces visibly worse assemblies. Every pipeline you will ever meet starts here. And most importantly, you have now seen what a settings file looks like and where the numbers live, so when a real one tells you 40% of your reads were discarded, you will actually notice instead of moving happily on to the next step.
</div>

**Exercise 20:** Tidy up and move on.

1. Make a new directory one level up called `trimmed_reads`. Remember `..` means one level up.
2. Move every file matching `*pair*` from `reads` into it.

In [ ]:
#add code for exercise 20 below this comment:

In [ ]:
#Check cell for exercise 20: run this as-is.
n=$(ls ../trimmed_reads/* 2>/dev/null | wc -l)
echo "files in ../trimmed_reads: $n  (expected 20)"
[ "$n" -eq 20 ] && echo "PASS" || echo "FAIL"

**Helper cell.** Run the cell below as-is. It:

- deletes the leftover files `AdapterRemoval` made that we do not need
- moves you into `trimmed_reads`
- adds `.fq` to the end of each trimmed file name, because the assembler recognises read files by their extension

Read it before you run it. Notice that the `rm` names each type of file explicitly rather than using a broad wildcard. That is deliberate: a `rm` command that says exactly what it deletes is a `rm` command that cannot surprise you.

In [ ]:
#Helper cell: run as-is.
cd ~/339_lab-main/reads
rm -f *.settings *.discarded *.collapsed *.collapsed.truncated *.singleton.truncated
cd ~/339_lab-main/trimmed_reads
for f in *.truncated
do
    mv "$f" "$f.fq"
done
echo "files now in trimmed_reads:"
ls

In [ ]:
#Check cell for the helper: run this as-is.
n=$(ls *.fq 2>/dev/null | wc -l)
echo "renamed files: $n  (expected 20)"
[ "$n" -eq 20 ] && echo "PASS" || echo "FAIL"

---
# Part 5: Assembling genomes

You now have clean reads. Time to turn them into genomes.

A **genome assembler** takes millions of short overlapping reads and reconstructs the long continuous sequences they came from. The usual analogy: take a stack of identical newspapers, shred them into 150 word fragments, and reconstruct the newspaper by finding fragments whose text overlaps. Assembly algorithms are super interesting, but we don't have time to go into how they work here. If you want to know more, let me know!

We will use `SPAdes`. For one pair of read files:

```bash
spades.py --pe1-1 fw_reads.fq --pe1-2 rv_reads.fq -o output_directory
```

The output directory must be different for every genome, or each assembly will overwrite the last.

**Exercise 21:** Write a script that assembles all ten genomes. You need to:

1. Build `fw_reads` and `rv_reads` arrays from the trimmed files. The trimmed forward reads match the pattern `*pair1*` and the reverse reads match `*pair2*`.
2. Loop over the positions.
3. At each position, assign `reads_1` and `reads_2` as before.
4. Make a directory to hold this assembly. Naming it after the loop position is easiest: `mkdir $i` makes a directory called `0`, then `1`, and so on.
5. Run `spades.py`, passing `-o $i` so each assembly goes into its own directory.

**Send the output to a log file.** SPAdes prints hundreds of lines per genome, and that much output can lock up the notebook. Add this to the end of your `spades.py` command:

```bash
> spades_$i.log 2>&1
```

That sends normal output and error messages into a log file instead of onto the screen. Add an `echo` before the SPAdes line so you can still see progress.

<div style="border-left:4px solid #c47f00;background:#fff8e6;padding:12px 16px;margin:14px 0">
<b>This cell takes a few minutes.</b> Around 30 to 60 seconds per genome. The <code>[*]</code> next to the cell means it is still running. Go and get a coffee.<br><br>
If it seems stuck for more than about 8 minutes, restart the kernel from the <b>Kernel</b> menu and run the check cell. If the check passes, the assemblies finished and only the notebook connection was stuck. Remember that a kernel restart wipes your variables and your working directory, so run the restore cell at the top of Session 2 first.
</div>

In [ ]:
#Run this cell first if you have restarted the kernel:
cd ~/339_lab-main/trimmed_reads
pwd

In [ ]:
#add code for exercise 21 below this comment:

In [ ]:
#Check cell for exercise 21: run this as-is.
cd ~/339_lab-main/trimmed_reads
n=$(ls [0-9]/contigs.fasta 2>/dev/null | wc -l)
echo "assemblies with a contigs.fasta: $n  (expected 10)"
if [ "$n" -eq 10 ]
then
    echo "PASS"
else
    echo "FAIL: check the spades_*.log files for errors, e.g.  tail -n 20 spades_0.log"
fi

**Helper cell.** Run as-is. It makes an `assemblies` directory, copies each assembly into it under a sensible name, and prints the first line of each one.

That first line is a FASTA header written by SPAdes, and it is worth reading. `NODE_1_length_65499_cov_24.0` tells you this is the longest contig, that it is 65,499 bases long, and that it was covered by reads about 24 times over. Coverage around 24x with a single long contig per genome means these assemblies are clean.

In [ ]:
#Helper cell: run as-is.
mkdir -p ../assemblies
fw_reads=(*pair1*)
for i in ${!fw_reads[@]}
do
    file_name=${fw_reads[$i]}
    new_name="${file_name%%.*}.fna"
    contig_file=$i/contigs.fasta
    printf "%-14s <- %s   " "$new_name" "$contig_file"
    head -n 1 $contig_file
    cp $contig_file ../assemblies/$new_name
done

**Expected output:** ten lines, each naming a genome and showing a `NODE_1_length_..._cov_24...` header. The exact lengths and coverages will vary slightly with your SPAdes version, which is completely normal.

<div style="border-left:4px solid #3b5bdb;background:#edf2ff;padding:16px 20px;margin:16px 0">
<h3 style="margin:0 0 10px">AI exercise 3: are these assemblies any good?</h3>

<b>The job.</b> I told you a moment ago that these assemblies look clean. You have no way at all of checking that, and "the lecturer said so" isn't much of a standard to hold your own data to.<br><br>

An assembler takes reads and gives you back <b>contigs</b>, which are the continuous stretches it managed to piece together. A really good bacterial assembly is one contig per chromosome. A bad one is thousands of fragments. So what you want to know is: how many contigs per genome, how long are they altogether, and how big is the biggest?<br><br>

There's also a standard summary statistic called <b>N50</b>, which you probably haven't met. Sort your contigs from longest to shortest, then add up their lengths until you get past half of the total assembly. Whichever contig you are on at that point is the N50. It tells you how long the contigs are that most of your genome actually lives in, which an average doesn't, because one enormous contig plus a thousand tiny ones gives you a lovely mean and a terrible assembly.<br><br>

Writing this yourself needs <code>awk</code>, which is an entire language and not one we are teaching today. So don't write it.<br><br>

<b>1. Ask.</b><br>
<i>"I have ten FASTA files of assembled bacterial contigs in a directory, all ending in .fna. Give me a BASH command that prints one row per file with: number of contigs, total assembly length, longest contig, and N50. Calculate the lengths from the sequences, not from the FASTA headers."</i><br><br>

That last sentence matters. SPAdes helpfully writes the length into its header lines, so a chatbot may well take the shortcut and read it from there, which then falls over on any FASTA that didn't come out of SPAdes. Asking for the general version gets you something you can reuse for the rest of your career.<br><br>

<b>2. Read it, then run it</b> in the cell below.<br><br>

<b>3. Check the output makes sense, and this one is a real result.</b> Total assembly length should be somewhere around 65,000 to 75,000 bases for every genome. How many of your ten came out as a <b>single</b> contig? Where a genome is one contig, its N50 and its longest contig and its total length are all the same number, which is a nice way of spotting it at a glance.<br><br>

<b>4. Get it to explain.</b> Ask these one at a time:<br>
<i>"What does <code>/^&gt;/</code> mean in this awk command, and what does the block after it do?"</i><br>
<i>"Walk me through the lines that work out the N50, using a made up set of five contig lengths."</i><br><br>

<b>Write it down.</b> Add a comment to the cell recording how many of your genomes came out as a single contig, and one sentence saying what N50 measures.
</div>

In [ ]:
#AI exercise 3: paste the assembly summary command here.
#Then add two comment lines: how many single-contig assemblies you got, and what N50 measures.
cd ~/339_lab-main/assemblies

---
<div style="border:2px solid #2b8a3e;background:#ebfbee;padding:16px 20px;margin:18px 0">
<h2 style="margin-top:0">End of Session 1</h2>

You have gone from raw sequencing reads to ten assembled genomes, which is not bad for three hours. You can now:

<ul>
<li>navigate a filesystem and move, copy, and delete files from a command line</li>
<li>use wildcards, pipes, and redirection</li>
<li>look inside a large data file without opening it</li>
<li>store lists of files in arrays and process them with a loop</li>
<li>drive real bioinformatics software from a script</li>
<li>get a chatbot to write you something past your own level, check it, run it, and then work out how it did it</li>
</ul>

<b>Before you leave:</b> save the notebook with Ctrl+S, and email a copy to yourself if there's any chance you'll be on a different machine next time.

<b>If you didn't finish:</b> that's completely fine. Just tell a demonstrator where you got to.
</div>

<div style="border-left:4px solid #6c757d;background:#f1f3f5;padding:14px 18px;margin:16px 0">
<b>Optional extensions, if you finished early</b><br><br>
<b>E1.</b> The assemblies are in <code>../assemblies</code>. Write a one line command using <code>grep -c</code> and a wildcard that counts the number of contigs in every assembly at once. (FASTA headers start with <code>&gt;</code>.)<br><br>
<b>E2.</b> Look at one of the <code>spades_*.log</code> files with <code>tail -n 30</code>. Find where it reports the k-mer sizes it used, then ask an AI what a k-mer is and why an assembler would try several sizes rather than picking one.<br><br>
<b>E3.</b> Take the read count command from AI exercise 1 and extend it: get it to print the <i>average read length</i> as well. Same routine, ask, read, run, then make it explain the part you did not recognise.
</div>

In [ ]:
#Space for the optional extensions:

---
# Session 2
---

## Start here

Two things before anything else.

**1. Reactivate your environment.** In your Ubuntu terminal, before launching JupyterLab:

```bash
conda activate biol339
cd ~/339_lab-main
jupyter lab
```

**2. Restore your session.** Restarting the kernel wipes every variable and resets the working directory. Run the cell below to put yourself back where Session 1 finished. Run it any time the kernel restarts.

In [ ]:
#Restore state: run this at the start of Session 2, and after any kernel restart.
cd ~/339_lab-main/assemblies
echo "working directory: $PWD"
echo "environment:       ${CONDA_DEFAULT_ENV:-NONE, you have not activated biol339}"
echo "assemblies found:  $(ls *.fna 2>/dev/null | wc -l)  (expected 10)"
ls

<div style="border-left:4px solid #c92a2a;background:#fff5f5;padding:12px 16px;margin:14px 0">
<b>If that showed fewer than 10 assemblies</b>, you did not finish Session 1. Do not carry on. Talk to a demonstrator, who can get you a set of assemblies to work from so you can do today's session.
</div>

---
# Part 6: Finding the genes

You have ten genome sequences. A genome sequence on its own tells you almost nothing. What you want to know is where the **genes** are, because genes are what get transcribed, and transcription is what we are going to measure.

We will use `glimmer3`, a gene finder for bacterial genomes. It works by learning what genes look like *in this particular genome* and then scanning for more of them. That takes four programs run in sequence:

| Step | Program | What it does |
|---|---|---|
| 1 | `long-orfs` | find long open reading frames, which are almost certainly real genes |
| 2 | `extract` | pull out their DNA sequences to use as training data |
| 3 | `build-icm` | build a statistical model of what a gene looks like in this genome |
| 4 | `glimmer3` | use that model to find all the genes, including the short ones |

Then `extract` runs once more to pull out the sequences of everything `glimmer3` found.

Assuming a variable `genome` holds a file name such as `genome_1.fna`, the full recipe is:

```bash
long-orfs $genome long_orfs.txt
extract -t $genome long_orfs.txt > run1.train
build-icm -r run1.icm < run1.train
glimmer3 $genome run1.icm run1
extract -t $genome run1.predict > genes_$genome
rm -f run1* long_orfs.txt
```

Line by line:

1. `long-orfs` reads the genome and writes the coordinates of long ORFs into `long_orfs.txt`.
2. `extract -t` reads those coordinates and writes the corresponding DNA sequences into `run1.train`.
3. `build-icm -r` reads the training sequences and writes a model to `run1.icm`.
4. `glimmer3` scans the genome using the model and writes gene coordinates to `run1.predict`.
5. `extract -t` turns those coordinates into sequences, in a file named `genes_` plus the genome file name.
6. `rm -f` clears the temporary files so the next genome starts clean.

<div style="border-left:4px solid #3b5bdb;background:#edf2ff;padding:12px 16px;margin:14px 0">
<b>A new symbol:</b> <code>&lt;</code> in <code>build-icm -r run1.icm &lt; run1.train</code>. You have met <code>&gt;</code>, which sends a program's output into a file. <code>&lt;</code> is the mirror image: it feeds the contents of a file into a program's input. Some older tools, like <code>build-icm</code>, expect their input this way instead of as a named argument.
</div>

<div style="border-left:4px solid #c47f00;background:#fff8e6;padding:12px 16px;margin:14px 0">
<b>Note that <code>$genome</code> already ends in <code>.fna</code>.</b> The array you are about to build holds full file names, so do not add another <code>.fna</code> anywhere. If you end up with a file called <code>genome_1.fna.fna</code>, that is what happened.
</div>

**Exercise 22:** Write a script that annotates all ten genomes.

1. Build an array called `genomes` from all files in this directory matching `*.fna`.
2. Loop over it. Since you need the value and not the position, you can loop over the values directly: `for genome in ${genomes[@]}`.
3. Inside the loop, run the six lines above.

The result should be ten new files: `genes_genome_1.fna` through `genes_genome_10.fna`.

**Build it up in pieces.** Write the loop with a single `echo $genome` inside first and check it prints ten file names. Then add the real commands. Testing a loop's plumbing before you put anything expensive inside it is a habit worth having.

`long-orfs` prints warnings about genetic codes and start codons. Those are normal, ignore them.

In [ ]:
#add code for exercise 22 below this comment:

In [ ]:
#Check cell for exercise 22: run this as-is.
n=$(ls genes_genome_*.fna 2>/dev/null | wc -l)
echo "gene files: $n  (expected 10)"
echo
echo "genes found per genome:"
grep -c "^>" genes_genome_*.fna 2>/dev/null
echo
bad=$(ls *.fna.fna 2>/dev/null | wc -l)
if [ "$n" -eq 10 ] && [ "$bad" -eq 0 ]
then
    echo "PASS"
else
    [ "$bad" -gt 0 ] && echo "FAIL: you have files ending .fna.fna, so a .fna got added twice."
    [ "$n" -ne 10 ] && echo "FAIL: expected 10 files named genes_genome_*.fna"
fi

**Expected output:** ten files, each containing roughly 40 to 85 genes. The exact counts depend on your software versions and do not need to match anyone else's.

**Helper cell.** Run as-is. It does two things:

1. **Renames the genes.** Right now every file has genes called `orf00001`, `orf00002`, and so on, so there are ten different genes called `orf00001` and no way to tell them apart. `sed` rewrites each FASTA header to include the file it came from, turning `>orf00042` into `>genes_genome_9_orf00042`.
2. **Concatenates** all ten files into one, `all_genes.fna`, which is what we will map reads against.

`sed` is a stream editor. `sed "s/^>/>PREFIX_/" file` means: substitute (`s`), at the start of a line (`^`), the character `>` with `>PREFIX_`. It reads the file and writes the edited version to its output, leaving the original alone, which is why we redirect that output with `>>` to build up `all_genes.fna` one genome at a time.

Note the `rm -f all_genes.fna` on the first line. Because we are appending with `>>`, the file has to be cleared first, otherwise running the cell twice would give you 1300 genes. Making a cell safe to re-run is a small habit that saves a lot of confusion.

In [ ]:
#Helper cell: run as-is.
rm -f all_genes.fna
for f in genes_genome_*.fna
do
    sed "s/^>/>${f%.fna}_/" "$f" >> all_genes.fna
done
echo "total genes in all_genes.fna: $(grep -c '^>' all_genes.fna)"
echo
echo "first three headers:"
grep "^>" all_genes.fna | head -n 3

**Expected output:** about 650 genes in total, with headers that now start `>genes_genome_1_orf00001` and so on.

---
# Part 7: Measuring gene expression

## What the patient data are

`patient_data` holds RNAseq reads from the gut microbiomes of 12 patients, two files each (forward and reverse). Six patients stayed healthy. Six developed colorectal cancer within two years of sampling.

**A transcriptome** is the full set of RNA transcripts being produced by an organism at a given moment. It tells you not what an organism *can* do, which is what its genome tells you, but what it *is currently doing*.

**A metatranscriptome** is the same idea for a whole community. Our data come from RNA extracted directly from stool samples, so every gut microbe in that patient contributed. (These data are imaginary, so thankfully nobody had to do that extraction.)

The logic of what follows: we have the genes of all ten gut species from Part 6. If we count how many RNA reads from each patient match each gene, we get a measure of how strongly each gene was being expressed in that patient's gut. Compare those counts between the healthy and cancer groups and you can find genes that behave differently, which is a lead worth chasing.

## Building an index

We will use `kallisto`, which maps RNAseq reads to a set of reference sequences very fast. Before it can map anything it has to convert the reference into an **index**, a data structure built for rapid lookup. Think of the index at the back of a textbook: same information as the book, reorganised so you can find things without reading every page.

```bash
kallisto index -i kallisto_index gene_file.fna --make-unique
```

- `-i kallisto_index` names the index file to create
- `gene_file.fna` is the input
- `--make-unique` handles duplicate gene names by adding a suffix, which matters because glimmer will happily call the same gene name in two genomes

**Exercise 23:** Run that command on the combined gene file you made at the end of Part 6.

In [ ]:
#add code for exercise 23 below this comment:

In [ ]:
#Check cell for exercise 23: run this as-is.
if [ -s kallisto_index ]
then
    echo "PASS: kallisto_index created, $(du -h kallisto_index | cut -f1) in size"
else
    echo "FAIL: kallisto_index is missing or empty."
fi

**Exercise 24:** Change directory to `patient_data` and list what is in it.

**Note the naming:** every forward read file ends in `_1.fq` and every reverse read file ends in `_2.fq`. That is the pattern you will build arrays from.

In [ ]:
#add code for exercise 24 below this comment:

**Exercise 25:** Make two arrays, `fw_reads` and `rv_reads`, holding the forward and reverse patient read files.

Be careful with your patterns. `*1*` would match `patient_11_2.fq`, which is a reverse read. Match on the ending instead.

In [ ]:
#add code for exercise 25 below this comment:

In [ ]:
#Check cell for exercise 25: run this as-is.
echo "fw_reads: ${#fw_reads[*]} files, last is ${fw_reads[11]}"
echo "rv_reads: ${#rv_reads[*]} files, last is ${rv_reads[11]}"
if [ "${#fw_reads[*]}" -eq 12 ] && [ "${#rv_reads[*]}" -eq 12 ] \
   && [ "${fw_reads[11]}" = "patient_9_1.fq" ] && [ "${rv_reads[11]}" = "patient_9_2.fq" ]
then
    echo "PASS"
else
    echo "FAIL: expected 12 files in each array, ending with patient_9_1.fq and patient_9_2.fq."
    echo "      If you got 24 in one array, your pattern is matching both directions."
fi

## Mapping the reads

To map one patient:

```bash
kallisto quant -i ../assemblies/kallisto_index -o $out_dir $fw_file $rv_file
```

- `-i` points at the index you just built
- `-o` is the directory to write results into, which must be different for every patient
- the two file names at the end are the forward and reverse reads. Unlike most arguments these have no flag, they are just positional, and kallisto expects forward first.

The output for each patient includes `abundance.tsv`, a table with one row per gene and a count of how many reads mapped to it.

**Exercise 26:** Write a script that runs `kallisto quant` on all 12 patients.

At each position in the loop you need to:

1. assign `fw_file` and `rv_file` from the two arrays
2. build an output directory name, then make that directory
3. run `kallisto quant`

For the output directory, strip the `_1.fq` off the forward file name so you get a clean patient name:

```bash
out_dir=../outputs/${fw_file%_1.fq}
mkdir -p $out_dir
```

That gives `../outputs/patient_1`, `../outputs/patient_2`, and so on. `mkdir -p` creates parent directories as needed and does not complain if the directory already exists, which makes the cell safe to re-run.

<div style="border-left:4px solid #c92a2a;background:#fff5f5;padding:12px 16px;margin:14px 0">
<b>The classic bug in this exercise.</b> It is very easy to write <code>rv_file=${fw_reads[$i]}</code> when you meant <code>rv_reads</code>. Everything runs, nothing errors, and you have quietly mapped the forward reads twice for every patient. Check that line character by character before you run it. Silent failures like this are far more dangerous than crashes, because the output looks perfectly reasonable.
</div>

In [ ]:
#add code for exercise 26 below this comment:

In [ ]:
#Check cell for exercise 26: run this as-is.
d=$(ls -d ~/339_lab-main/outputs/*/ 2>/dev/null | wc -l)
t=$(ls ~/339_lab-main/outputs/*/abundance.tsv 2>/dev/null | wc -l)
echo "output directories: $d  (expected 12)"
echo "abundance.tsv files: $t  (expected 12)"
echo
echo "rows in each table (should all be identical):"
wc -l ~/339_lab-main/outputs/*/abundance.tsv | head -n 13
echo
if [ "$d" -eq 12 ] && [ "$t" -eq 12 ]
then
    echo "PASS: all 12 patients quantified."
else
    echo "FAIL: check the loop and re-run."
fi

## What kallisto gave you

Run the cell below to look at the top of one patient's results.

| Column | Meaning |
|---|---|
| `target_id` | the gene name, which you built yourself in Part 6 |
| `length` | the length of the gene in bases |
| `eff_length` | effective length, adjusted because a fragment cannot start in the last few bases of a gene |
| `est_counts` | the estimated number of reads assigned to this gene |
| `tpm` | transcripts per million, an expression measure normalised for gene length and sequencing depth |

**A question to carry into the Python notebook.** Two of those columns measure expression: `est_counts` and `tpm`. Only one of them is the right input to a differential expression test, and the other one will happily produce a table of p-values that you should not trust.

Have a guess at which, and why, before you open `python_lab.ipynb`. It answers the question properly, and the reasoning is more useful than the code.

In [ ]:
head -n 5 ~/339_lab-main/outputs/patient_1/abundance.tsv

---
# Part 8: Over to the Python notebook

You have finished the command line half of this lab. What you have built is a real, if small, metatranscriptomics pipeline:

```
raw reads  ->  trimming  ->  assembly  ->  gene finding  ->  read mapping  ->  count tables
```

Twelve count tables are now sitting in `~/339_lab-main/outputs/`. The last step is to compare them statistically, and that is easier in Python, so we change tools.

## What to do now

1. In the JupyterLab file browser on the left, open **`python_lab.ipynb`**.
2. Check the kernel in the **top right says `Python 3`**, not Bash. If it says Bash, click it and change it.
3. Work through it. It is short. Most of the code is written for you, and the interesting part is the interpretation at the end.

<div style="border:2px solid #2b8a3e;background:#ebfbee;padding:16px 20px;margin:18px 0">
<h3 style="margin-top:0">What you learned here</h3>
Every one of these transfers directly to any command line bioinformatics you meet later:
<ul>
<li>the shell, the working directory, and why <code>pwd</code> is the answer to most confusion</li>
<li>wildcards, and the fact that the shell expands them before the command ever sees them</li>
<li>pipes, redirection, and reading a large file without opening it</li>
<li>variables, arrays, and loops, which is how one command becomes a pipeline over 10 or 10,000 samples</li>
<li>running published bioinformatics tools, reading their logs, and checking their output</li>
<li>using AI to reach past what you have been taught, and verifying what it gives you before trusting it</li>
</ul>
The specific tools will be different in whatever lab you end up in. The shape of the work will be exactly this.
</div>

In [ ]:
#Space for notes: